# Deploy NVIDIA Parakeet Realtime EOU 120M on Vertex AI

This notebook deploys the [NVIDIA Parakeet Realtime EOU 120M](https://huggingface.co/nvidia/parakeet_realtime_eou_120m-v1) model on **Google Cloud Vertex AI** for online prediction.

| | |
|---|---|
| **Task** | Streaming ASR with End-of-Utterance (EOU) detection |
| **Architecture** | FastConformer-RNNT (Cache-Aware Streaming) |
| **Parameters** | 120 M |
| **Language** | English |
| **Input** | 16 kHz mono WAV audio |
| **Output** | Transcription with `<EOU>` tokens marking utterance boundaries |
| **License** | NVIDIA Open Model License |

**What this notebook does:**
1. Builds a custom serving container with the model baked in
2. Pushes it to Google Artifact Registry via Cloud Build
3. Deploys the model to a Vertex AI Endpoint on an NVIDIA T4 GPU
4. Sends sample audio for transcription and shows results
5. Cleans up all resources

## Prerequisites

Before running this notebook, ensure you have:

1. A **Google Cloud project** with billing enabled
2. The following **APIs enabled** (click the links to enable):
   - [Vertex AI API](https://console.cloud.google.com/apis/api/aiplatform.googleapis.com)
   - [Artifact Registry API](https://console.cloud.google.com/apis/api/artifactregistry.googleapis.com)
   - [Cloud Build API](https://console.cloud.google.com/apis/api/cloudbuild.googleapis.com)
3. Your account has the following **IAM roles**:
   - `Vertex AI User` (or Admin)
   - `Artifact Registry Writer`
   - `Cloud Build Editor`
   - `Storage Admin` (for staging)

In [ ]:
!pip install -q google-cloud-aiplatform gTTS pydub

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated via Colab OAuth.")
else:
    print(
        "Not running in Colab.\n"
        "Make sure Application Default Credentials are set:\n"
        "  gcloud auth application-default login"
    )

In [ ]:
# ============================================================
#  CONFIGURATION — update these for your project
# ============================================================
PROJECT_ID = "your-project-id"   # @param {type:"string"}
REGION     = "us-central1"       # @param {type:"string"}

REPOSITORY = "ml-serving"        # Artifact Registry repo name
IMAGE_NAME = "parakeet-eou-serving"
IMAGE_TAG  = "v1"

# Derived ─ no changes needed below
CONTAINER_IMAGE_URI = (
    f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPOSITORY}/{IMAGE_NAME}:{IMAGE_TAG}"
)
ENDPOINT_DISPLAY_NAME = "parakeet-eou-endpoint"
MODEL_DISPLAY_NAME    = "parakeet-realtime-eou-120m"

# Point gcloud at the right project
!gcloud config set project {PROJECT_ID} --quiet

# Initialise Vertex AI SDK
from google.cloud import aiplatform
aiplatform.init(project=PROJECT_ID, location=REGION)

print(f"Project:   {PROJECT_ID}")
print(f"Region:    {REGION}")
print(f"Container: {CONTAINER_IMAGE_URI}")

## Step 1 — Build the Custom Serving Container

We create a Docker image that:
1. Starts from the NVIDIA PyTorch base image (CUDA 12 + PyTorch)
2. Installs NeMo ASR toolkit and audio libraries
3. **Pre-downloads** the Parakeet model so container startup is fast
4. Serves predictions via a lightweight Flask HTTP server

The container accepts **base64-encoded WAV audio** (16 kHz, mono) and returns
transcription text with `<EOU>` markers.

In [ ]:
import os

BUILD_DIR = "/content/parakeet_serving"
os.makedirs(BUILD_DIR, exist_ok=True)
print(f"Build directory: {BUILD_DIR}")

In [ ]:
%%writefile /content/parakeet_serving/handler.py
"""
Vertex AI prediction handler for NVIDIA Parakeet Realtime EOU 120M.
Routes
------
GET  /health   – liveness / readiness probe
POST /predict  – transcribe audio instances
"""
import os, sys, base64, tempfile, logging
import torch
from flask import Flask, request, jsonify
logging.basicConfig(
    level=logging.INFO,
    stream=sys.stdout,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger(__name__)
# -------------------------------------------------------------------
# Model loading — runs once when gunicorn imports this module
# -------------------------------------------------------------------
logger.info("Loading NVIDIA Parakeet Realtime EOU 120M model ...")
import nemo.collections.asr as nemo_asr
MODEL_NAME = "nvidia/parakeet_realtime_eou_120m-v1"
model = nemo_asr.models.ASRModel.from_pretrained(model_name=MODEL_NAME)
model.eval()
if torch.cuda.is_available():
    model = model.cuda()
    logger.info("Model loaded on GPU: %s", torch.cuda.get_device_name(0))
else:
    logger.warning("No GPU detected — inference will be slow.")
logger.info("Model ready for inference.")
# -------------------------------------------------------------------
# Flask application
# -------------------------------------------------------------------
app = Flask(__name__)
AIP_HEALTH_ROUTE  = os.environ.get("AIP_HEALTH_ROUTE", "/health")
AIP_PREDICT_ROUTE = os.environ.get("AIP_PREDICT_ROUTE", "/predict")
@app.route(AIP_HEALTH_ROUTE, methods=["GET"])
def health():
    """Vertex AI health / readiness probe."""
    return jsonify({"status": "healthy"}), 200
@app.route(AIP_PREDICT_ROUTE, methods=["POST"])
def predict():
    """Transcribe base64-encoded 16 kHz mono WAV audio.
    Request body
    ------------
    {
        "instances": [
            {"audio_base64": "<base64-encoded WAV bytes>"}
        ]
    }
    Response body
    -------------
    {
        "predictions": [
            {
                "transcription": "hello how are you<EOU>",
                "end_of_utterance_detected": true
            }
        ]
    }
    """
    data = request.get_json(force=True)
    instances = data.get("instances", [])
    if not instances:
        return jsonify({"error": "No instances provided."}), 400
    predictions = []
    for instance in instances:
        audio_b64 = instance.get("audio_base64", "")
        if not audio_b64:
            predictions.append(
                {"transcription": "", "end_of_utterance_detected": False,
                 "error": "Missing audio_base64."}
            )
            continue
        # Write decoded audio to a temp WAV file
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
            f.write(base64.b64decode(audio_b64))
            tmp = f.name
        try:
            output = model.transcribe([tmp])
            text = (
                output[0].text
                if hasattr(output[0], "text")
                else str(output[0])
            )
            predictions.append({
                "transcription": text,
                "end_of_utterance_detected": "<EOU>" in text or "<eou>" in text,
            })
        except Exception as exc:
            logger.error("Transcription failed: %s", exc, exc_info=True)
            predictions.append(
                {"transcription": "", "end_of_utterance_detected": False,
                 "error": str(exc)}
            )
        finally:
            os.unlink(tmp)
    return jsonify({"predictions": predictions})
if __name__ == "__main__":
    port = int(os.environ.get("AIP_HTTP_PORT", 8080))
    app.run(host="0.0.0.0", port=port)

In [ ]:
%%writefile /content/parakeet_serving/Dockerfile
FROM nvcr.io/nvidia/pytorch:24.01-py3

# System deps for audio processing
RUN apt-get update && \
    apt-get install -y --no-install-recommends libsndfile1 ffmpeg sox && \
    rm -rf /var/lib/apt/lists/*

# Python deps — NeMo ASR toolkit + serving stack
# NeMo pins its own PyTorch version, so install it first.
RUN pip install --no-cache-dir \
    Cython packaging \
    flask gunicorn soundfile \
    "nemo_toolkit[asr]"

# Install torchvision and torchaudio AFTER NeMo so pip resolves versions
# that match the PyTorch version NeMo installed.
# Pin setuptools<78 because 78+ removed pkg_resources, which NeMo needs.
RUN pip install --no-cache-dir torchvision torchaudio "setuptools<78"

# Pre-download the model during build so startup is fast
RUN python -c "\
import nemo.collections.asr as nemo_asr; \
m = nemo_asr.models.ASRModel.from_pretrained( \
    'nvidia/parakeet_realtime_eou_120m-v1'); \
del m; print('Model cached.')"

# Copy serving handler
COPY handler.py /app/handler.py
WORKDIR /app

# Vertex AI injects these env vars at runtime
ENV AIP_HTTP_PORT=8080
ENV AIP_HEALTH_ROUTE=/health
ENV AIP_PREDICT_ROUTE=/predict

EXPOSE 8080

CMD exec gunicorn \
    --bind 0.0.0.0:${AIP_HTTP_PORT} \
    --timeout 300 \
    --workers 1 \
    --threads 2 \
    handler:app

In [ ]:
# Create the Artifact Registry repository (idempotent)
!gcloud artifacts repositories create {REPOSITORY} \
    --repository-format=docker \
    --location={REGION} \
    --description="ML model serving containers" \
    --quiet 2>/dev/null || true

# Build & push the container image via Cloud Build.
# This typically takes 20-40 minutes (NeMo install + model download).
print(f"Building container: {CONTAINER_IMAGE_URI}")
print("This will take 20-40 min — grab a coffee.\n")

!cd /content/parakeet_serving && gcloud builds submit \
    --tag {CONTAINER_IMAGE_URI} \
    --timeout=3600 \
    --machine-type=e2-highcpu-8

## Step 2 — Deploy to Vertex AI

We register the container as a Vertex AI **Model**, create an **Endpoint**,
and deploy the model on an **NVIDIA T4 GPU** (`n1-standard-4`).

The T4 has 16 GB VRAM — far more than the ~0.5 GB this 120 M-parameter
model needs — and is the most **cost-effective** GPU on Vertex AI.

In [ ]:
from google.cloud import aiplatform

# Upload model to Vertex AI Model Registry
model = aiplatform.Model.upload(
    display_name=MODEL_DISPLAY_NAME,
    description=(
        "NVIDIA Parakeet Realtime EOU 120M — "
        "streaming ASR with end-of-utterance detection"
    ),
    serving_container_image_uri=CONTAINER_IMAGE_URI,
    serving_container_predict_route="/predict",
    serving_container_health_route="/health",
    serving_container_ports=[8080],
    sync=True,
)
model.wait()
print(f"Model uploaded: {model.resource_name}")

# Create an endpoint
endpoint = aiplatform.Endpoint.create(
    display_name=ENDPOINT_DISPLAY_NAME,
)
print(f"Endpoint created: {endpoint.resource_name}")

# Deploy model to endpoint
print("\nDeploying model — this typically takes 10-15 minutes ...")
model.deploy(
    endpoint=endpoint,
    deployed_model_display_name=MODEL_DISPLAY_NAME,
    machine_type="n1-standard-4",
    accelerator_type="NVIDIA_TESLA_T4",
    accelerator_count=1,
    min_replica_count=1,
    max_replica_count=1,
    traffic_percentage=100,
    deploy_request_timeout=3600,
    sync=True,
)

print(f"\nModel deployed!")
print(f"Endpoint: {endpoint.resource_name}")

## Step 3 — Test the Deployed Model

We generate speech audio using Google Text-to-Speech, convert it to the
format the model expects (16 kHz, mono, WAV), and send it to the endpoint.

In [ ]:
import base64
from gtts import gTTS
from pydub import AudioSegment

def text_to_base64_wav(text: str, path: str = "/tmp/tts.wav") -> str:
    """Convert text to a base64-encoded 16 kHz mono WAV via gTTS."""
    tts = gTTS(text, lang="en")
    mp3_path = path.replace(".wav", ".mp3")
    tts.save(mp3_path)
    audio = (
        AudioSegment.from_mp3(mp3_path)
        .set_frame_rate(16000)
        .set_channels(1)
        .set_sample_width(2)  # 16-bit
    )
    audio.export(path, format="wav")
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

# Single-utterance test
test_text = "Hello, what is the weather like today?"
audio_b64 = text_to_base64_wav(test_text)
print(f'Test audio generated: "{test_text}"')

In [ ]:
# Send to the deployed endpoint
response = endpoint.predict(instances=[{"audio_base64": audio_b64}])

print("=" * 60)
print("PREDICTION RESULT")
print("=" * 60)
for pred in response.predictions:
    print(f"  Transcription : {pred['transcription']}")
    print(f"  EOU detected  : {pred['end_of_utterance_detected']}")

In [ ]:
# Batch test with multiple utterances
test_phrases = [
    "My name is John and I work at a technology company.",
    "Can you book me a flight to New York?",
    "The quick brown fox jumps over the lazy dog.",
]

instances = [
    {"audio_base64": text_to_base64_wav(p, f"/tmp/tts_{i}.wav")}
    for i, p in enumerate(test_phrases)
]

response = endpoint.predict(instances=instances)

print("=" * 60)
print("BATCH PREDICTION RESULTS")
print("=" * 60)
for phrase, pred in zip(test_phrases, response.predictions):
    print(f"\n  Input:         \"{phrase}\"")
    print(f"  Transcription: \"{pred['transcription']}\"")
    print(f"  EOU detected:  {pred['end_of_utterance_detected']}")

### (Optional) Test with your own audio

Upload a **WAV file** (16 kHz, mono recommended) and send it to the endpoint.

In [ ]:
import base64

# Upload a WAV file from your local machine
from google.colab import files
uploaded = files.upload()  # pick a .wav file

for filename, content in uploaded.items():
    audio_b64 = base64.b64encode(content).decode("utf-8")
    response = endpoint.predict(instances=[{"audio_base64": audio_b64}])
    for pred in response.predictions:
        print(f"File: {filename}")
        print(f"  Transcription: {pred['transcription']}")
        print(f"  EOU detected:  {pred['end_of_utterance_detected']}")

## Step 4 — Clean Up Resources

**Important:** The deployed endpoint incurs charges (~\$0.35/hr for the T4
node). Run the cell below to delete everything when you are done.

In [ ]:
# Undeploy model from the endpoint
endpoint.undeploy_all()
print("Model undeployed.")

# Delete the endpoint
endpoint.delete()
print("Endpoint deleted.")

# Delete the model from the registry
model.delete()
print("Model deleted.")

print("\nAll Vertex AI resources cleaned up — no further charges.")

# (Optional) Delete the container image from Artifact Registry:
# !gcloud artifacts docker images delete {CONTAINER_IMAGE_URI} --quiet